[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/16_hyperparameter_optimization/90_exercises.ipynb)

# Bài tập — Chương 16: Hyperparameter Optimization (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/16_hyperparameter_optimization

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (GridSearchCV, ParameterSampler, RandomizedSearchCV, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** So sánh grid search, random search, successive halving, Hyperband, Bayesian optimization (GP + EI), TPE — mỗi phương pháp một câu: ý tưởng và khi nào dùng.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một bạn tune 2 000 cấu hình XGBoost trên một tập validation cố định 300 mẫu và tự hào báo cáo AUC 0.93 (baseline mặc định 0.86). Giải thích vì sao con số 0.93 đáng ngờ và đề xuất cách làm.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Successive halving: 81 cấu hình, ngân sách ban đầu 100 mẫu, $\eta = 3$, ngân sách tối đa 8 100. Liệt kê (số cấu hình, ngân sách) mỗi vòng và tổng "mẫu-train". So với chạy cả 81 cấu hình ở 8 100 mẫu thì tiết kiệm bao nhiêu lần?

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
rounds = None
total_sh = None
total_full = None
speedup = None
### END CODE HERE ###
print(rounds, total_sh, total_full, round(speedup, 1))

In [ ]:
check_equal(rounds, [(81, 100), (27, 300), (9, 900), (3, 2700), (1, 8100)], "các vòng")
check_equal(total_sh, 5 * 8100, "mỗi vòng tốn 8 100 → tổng 40 500")
check_close(speedup, 16.2, message="tiết kiệm 16.2 lần")

**Câu 4.** Expected Improvement tại một điểm có $\mu = 0.9$, $\sigma = 0.1$, giá trị tốt nhất hiện tại $f^* = 0.85$, $\xi = 0$. Tính $z$ và EI. So sánh với điểm $\mu = 0.8$, $\sigma = 0.3$ — điểm nào được chọn?

In [ ]:
from scipy.stats import norm

### START CODE HERE ### (≈ 4 dòng)
z1 = None
ei1 = None
z2 = None
ei2 = None
### END CODE HERE ###
print(f"EI1 = {ei1:.4f} | EI2 = {ei2:.4f}")

In [ ]:
check_close(z1, 0.5, message="z1")
check_close(ei1, 0.05 * norm.cdf(0.5) + 0.1 * norm.pdf(0.5), message="EI điểm 1")
check_true(ei2 > ei1, "điểm 2 (μ thấp hơn nhưng bất định lớn) có EI lớn hơn → khám phá")

## L3 — Code

**Bài 5.** Cài đặt `random_search(objective, space_sampler, n_iter, seed)` trả về `(best_params, best_score, history)` — `space_sampler(rng)` trả về một dict tham số. Áp dụng để tune `C` (log-uniform $10^{-4}$–$10^2$) của Logistic Regression trên churn.

In [ ]:
def random_search(objective, space_sampler, n_iter=20, seed=0):
    r = np.random.default_rng(seed)
    history = []
    ### START CODE HERE ### (≈ 5 dòng)
    # lặp n_iter lần: params = space_sampler(r); history.append((params, objective(params)))
    best_params, best_score = None, None     # phần tử có điểm cao nhất trong history
    ### END CODE HERE ###
    return best_params, best_score, history

In [ ]:
df = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
Xc = pd.get_dummies(df.drop(columns=["customer_id", "churn"]), dtype=float)
yc = df["churn"].to_numpy()
cv = StratifiedKFold(4, shuffle=True, random_state=0)


def lr_objective(params):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=params["C"], max_iter=3000))
    return cross_val_score(model, Xc, yc, cv=cv, scoring="roc_auc").mean()


best_p, best_s, hist = random_search(lr_objective, lambda r: {"C": 10 ** r.uniform(-4, 2)}, n_iter=15)
print(f"C tốt nhất = {best_p['C']:.4g} | ROC-AUC CV = {best_s:.4f}")
check_equal(len(hist), 15, "15 lần thử")
check_greater(best_s, lr_objective({"C": 1e-4}), "tốt hơn C cực nhỏ")

**Bài 6.** Cài đặt `successive_halving(configs, evaluate, min_budget, max_budget, eta)` tổng quát: `evaluate(config, budget)` trả về điểm (càng cao càng tốt). Trả về `(best_config, log)` với `log` là list `(budget, n_configs)` mỗi vòng.

In [ ]:
def successive_halving(configs, evaluate, min_budget, max_budget, eta=3):
    budget, log = min_budget, []
    ### START CODE HERE ### (≈ 8 dòng)
    # lặp: đánh giá mọi cấu hình ở `budget`; ghi log (budget, số cấu hình); nếu còn 1 cấu hình hoặc budget ≥ max_budget → trả về tốt nhất
    # ngược lại giữ max(1, len // eta) cấu hình tốt nhất, budget = min(budget * eta, max_budget)
    pass
    ### END CODE HERE ###

In [ ]:
# kiểm tra trên bài toán giả: điểm thật = −(x − 0.3)², đánh giá ở ngân sách b có nhiễu σ = 1/√b
toy_configs = [{"x": v} for v in np.linspace(0, 1, 27)]
toy_eval = lambda c, b: -(c["x"] - 0.3) ** 2 + np.random.default_rng(int(b * 1000 + c["x"] * 100)).normal(0, 0.5 / np.sqrt(b))
best_toy, log_toy = successive_halving(toy_configs, toy_eval, min_budget=1, max_budget=81, eta=3)
print(best_toy, log_toy)
check_equal(log_toy, [(1, 27), (3, 9), (9, 3), (27, 1)], "số cấu hình giảm 3 lần mỗi vòng")
check_less(abs(best_toy["x"] - 0.3), 0.1, "tìm được x gần 0.3")

## L4 — Debug

**Bài 7.** Đoạn tuning sau có **3 vấn đề**: (1) `learning_rate` lấy mẫu **uniform** trên [0.0001, 1] (gần như mọi mẫu > 0.1); (2) tune `max_iter` bằng search thay vì early stopping; (3) đánh giá model tốt nhất trên **chính tập dùng để tune**. Sửa: `loguniform`, bật `early_stopping=True` với `max_iter` lớn, và đánh giá trên tập test tách riêng (`test_auc`).

In [ ]:
from scipy.stats import uniform

X_dev, X_test, y_dev, y_test = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
### START CODE HERE ### (≈ 7 dòng)
space = {"learning_rate": uniform(0.0001, 1), "max_iter": randint(10, 500), "max_leaf_nodes": randint(4, 64)}   # 🐛 (1), (2)
search = RandomizedSearchCV(HistGradientBoostingClassifier(early_stopping=False, random_state=0), space, n_iter=10,
                            cv=3, scoring="roc_auc", random_state=0).fit(Xc, yc)
test_auc = search.best_score_                                                                                 # 🐛 (3)
### END CODE HERE ###
lr_samples = search.cv_results_["param_learning_rate"].astype(float)
print(f"learning_rate đã thử: {np.sort(lr_samples).round(4)} | ROC-AUC test = {test_auc:.4f}")

In [ ]:
check_greater(np.mean(lr_samples < 0.05), 0.2, "≥ 20% mẫu learning_rate < 0.05 (log-uniform)")
check_true("max_iter" not in search.best_params_, "không tune max_iter — dùng early stopping")
check_greater(test_auc, 0.78, "ROC-AUC test (tập riêng) ≥ 0.78")

## L5 — Phân tích

**Bài 8.** Lặp lại thí nghiệm "grid vs random" của bài 01 với hàm mục tiêu **3 tham số** trong đó chỉ **1** quan trọng: $f(a, b, c) = \exp(-(a - 0.37)^2/0.003) + 0.02(b + c)$. Với ngân sách 27 lần thử (grid 3×3×3 dịch ngẫu nhiên vs random), đo tỉ lệ tìm được $f > 0.9$ qua 500 lần lặp. Lưu `p_grid`, `p_random`. Nhận xét: lợi thế của random tăng hay giảm khi số tham số không quan trọng tăng?

In [ ]:
def f3(a, b, c):
    return np.exp(-((a - 0.37) ** 2) / 0.003) + 0.02 * (b + c)


### START CODE HERE ### (≈ 10 dòng)
# 500 lần lặp: grid 3×3×3 (3 giá trị mỗi chiều, dịch ngẫu nhiên) vs 27 điểm ngẫu nhiên; ghi lại max f3 > 0.9
p_grid, p_random = None, None
### END CODE HERE ###
print(f"tìm được đỉnh: grid {p_grid:.0%} | random {p_random:.0%}")

In [ ]:
check_greater(p_random, 2 * p_grid, "random tốt hơn hẳn grid khi chỉ 1/3 tham số quan trọng")

✍️ Nhận xét:

## L6 — Thực tế

**Bài 9.** Dùng Optuna (nếu có; nếu không, dùng `RandomizedSearchCV`) để chọn giữa **Logistic Regression** (tune `C`) và **HistGradientBoosting** (tune `learning_rate`, `max_leaf_nodes`, `min_samples_leaf`) trên `X_dev` (4-fold, ROC-AUC), 25 lần thử. Train lại cấu hình tốt nhất trên `X_dev`, báo cáo `final_test_auc` trên `X_test` và so với `best_cv` (điểm CV tốt nhất).

In [ ]:
def build_model(params):
    if params["model"] == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(C=params["C"], max_iter=3000))
    return HistGradientBoostingClassifier(learning_rate=params["lr"], max_leaf_nodes=params["leaves"],
                                          min_samples_leaf=params["min_leaf"], max_iter=300, early_stopping=True, random_state=0)


cv4 = StratifiedKFold(4, shuffle=True, random_state=1)
### START CODE HERE ### (≈ 18 dòng)
best_params, best_cv = None, None    # Optuna (TPE, 25 lần thử, không gian điều kiện) — hoặc RandomizedSearch nếu không có optuna
final_model = None                   # build_model(best_params).fit(X_dev, y_dev)
final_test_auc = None
### END CODE HERE ###
print("cấu hình tốt nhất:", best_params)
print(f"ROC-AUC: CV tốt nhất = {best_cv:.4f} | test = {final_test_auc:.4f}")

In [ ]:
check_greater(final_test_auc, 0.80, "ROC-AUC test ≥ 0.80")

## L7 — Nâng cao

**Bài 10.** Cài đặt **Bayesian optimization 1 chiều** for log10(C) của Logistic Regression trên `X_dev` (khoảng $[-4, 2]$): 3 điểm khởi đầu ngẫu nhiên + 7 bước, GP (`Matern` + `WhiteKernel` cố định nhỏ) và Expected Improvement trên lưới 200 điểm. Trả về `bo_best_logC`, `bo_best_score` và so với random search 10 điểm (`rs_best_score`).

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
import warnings

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


def score_logC(logC):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=10**logC, max_iter=3000))
    return cross_val_score(model, X_dev, y_dev, cv=cv4, scoring="roc_auc").mean()


grid10 = np.linspace(-4, 2, 200).reshape(-1, 1)
r10 = np.random.default_rng(5)
X_obs = r10.uniform(-4, 2, 3).reshape(-1, 1)
y_obs = np.array([score_logC(v) for v in X_obs.ravel()])
### START CODE HERE ### (≈ 9 dòng)
# 7 bước: fit GP lên (X_obs, y_obs) → μ, σ trên grid10 → EI → x_next = argmax EI → đánh giá score_logC(x_next), thêm vào X_obs, y_obs
bo_best_logC, bo_best_score = None, None
### END CODE HERE ###
rs_best_score = max(score_logC(v) for v in np.random.default_rng(6).uniform(-4, 2, 10))
print(f"BO: log10(C) = {bo_best_logC:.2f}, ROC-AUC = {bo_best_score:.4f} | random 10 điểm: {rs_best_score:.4f}")
check_greater(bo_best_score, rs_best_score - 0.002, "BO không thua random search (cùng 10 lần đánh giá)")
check_equal(len(y_obs), 10, "đúng 10 lần đánh giá")